In [ ]:
import sys, subprocess, importlib.util

required = {
    'openai': 'openai', 'pandas': 'pandas', 'numpy': 'numpy',
    'scipy': 'scipy', 'PIL': 'pillow', 'tqdm': 'tqdm',
    'IPython': 'ipython'
}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])


In [ ]:
from __future__ import annotations

import base64
import hashlib
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr
from tqdm.auto import tqdm
from openai import OpenAI

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    drive = userdata = None

pd.set_option('display.max_colwidth', 180)
pd.set_option('display.max_columns', 100)


In [ ]:
METRIC = 'CI'
TARGET_COLUMN = 'CRAI_CI'
MODEL = 'gpt-5.4'
REASONING_EFFORT = 'none'
IMAGE_DETAIL = 'original'
MAX_RETRIES = 3
REQUEST_SLEEP_SECONDS = 0.20

                                                   
RUN_DEV_API_CALLS = True
RUN_TEST_API_CALLS = True

if IN_COLAB:
    drive.mount('/content/drive')
    IMAGEEVAL_ROOT = Path('/content/drive/MyDrive/Dr. Lulwah - Ahmed/ImageEVAl')
else:
    IMAGEEVAL_ROOT = Path(os.environ.get('IMAGEEVAL_ROOT', '.')).resolve()

PROJECT_DIR = IMAGEEVAL_ROOT / 'ImageEval2026_Task2_CRAI_Bench'
DATA_CANDIDATES = [PROJECT_DIR / 'data', IMAGEEVAL_ROOT / 'train_dev']
DATA_DIR = next(
    (path for path in DATA_CANDIDATES
     if (path / 'dev' / 'captions.tsv').exists()),
    DATA_CANDIDATES[0],
)

EXPERIMENT_ROOT = PROJECT_DIR / 'ci_direct_gpt54_no_training_v1'
CACHE_DIR = EXPERIMENT_ROOT / 'cache'
OUTPUT_DIR = EXPERIMENT_ROOT / 'outputs'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Data:', DATA_DIR)
print('Experiment:', EXPERIMENT_ROOT)
print('Target:', TARGET_COLUMN)
print('Model:', MODEL, '| reasoning:', REASONING_EFFORT,
      '| detail:', IMAGE_DETAIL)
print('API switches:', RUN_DEV_API_CALLS, RUN_TEST_API_CALLS)


In [ ]:
def parse_base_id(instance_id: str) -> str:
    return re.sub(r'_v\d+$', '', str(instance_id))

def parse_version(instance_id: str) -> int:
    match = re.search(r'_v(\d+)$', str(instance_id))
    return int(match.group(1)) if match else -1

def caption_text(row: pd.Series) -> str:
    for column in ['caption', 'text', 'prompt']:
        if column in row and pd.notna(row[column]):
            return str(row[column])
    raise KeyError('No caption/text/prompt column found')

def find_image(folder: Path, stem: str) -> Path:
    for suffix in ['.png', '.jpg', '.jpeg', '.webp']:
        candidate = folder / f'{stem}{suffix}'
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'No image for {stem!r} in {folder}')

def load_inputs(split: str) -> pd.DataFrame:
    split_dir = DATA_DIR / split
    frame = pd.read_csv(split_dir / 'captions.tsv', sep='	')
    frame['id'] = frame['id'].astype(str)
    frame['split'] = split
    frame['base_id'] = frame['id'].map(parse_base_id)
    frame['caption_version'] = frame['id'].map(parse_version)
    frame['ref_image_path'] = frame['base_id'].map(
        lambda value: str(find_image(split_dir / 'imgs' / 'ref', value))
    )
    frame['generated_image_path'] = frame['id'].map(
        lambda value: str(find_image(split_dir / 'imgs' / 'generated', value))
    )
    return frame

def assert_groups(frame: pd.DataFrame, split: str):
    assert frame['id'].is_unique, f'{split}: duplicate IDs'
    versions = frame.groupby('base_id')['caption_version'].apply(
        lambda values: set(map(int, values))
    )
    bad = versions[versions != {1, 2, 3, 4, 5}]
    assert bad.empty, f'{split}: incomplete groups {bad.to_dict()}'
    assert frame.groupby('base_id')['ref_image_path'].nunique().eq(1).all()

dev_df = load_inputs('dev')
test_available = (DATA_DIR / 'test' / 'captions.tsv').exists()
test_df = load_inputs('test') if test_available else pd.DataFrame()

assert_groups(dev_df, 'dev')
if test_available:
    assert_groups(test_df, 'test')

print('dev:', len(dev_df), 'rows |', dev_df['base_id'].nunique(), 'groups')
print('test:', (f'{len(test_df)} rows | {test_df["base_id"].nunique()} groups'
                if test_available else 'not found'))


In [ ]:
PROMPT_VERSION = 'ci-direct-no-training-gpt54-v1'
SCHEMA_VERSION = 'ci-direct-score-schema-v1'
SYSTEM_PROMPT = 'ROLE\n\nYou are a strict multimodal evaluator for ImageEval 2026 CRAI-Bench. Judge only\nCRAI_CI: Cultural Integrity.\n\nTARGET\n\nJudge whether the generated image preserves the cultural identity and meaning of the\nreference scene. Look for generic replacement, culturally incorrect transformation,\nstereotyping, or distortion of important attire, practices, objects, architecture, and\nsocial meaning. Harmless changes in viewpoint, composition, pose, lighting, or visual style\nmust not reduce the score when the cultural identity remains intact.\n\nDo not turn this into CEA (counting present elements), CC (scene coherence), CS (degree of\nQatari specificity), HP (unsupported fabricated additions), or an image quality score. Use\nthe current caption and authentic reference image together to determine the cultural identity\nthat should be preserved.\n\nSCORE BANDS\n\n- 0.00--0.20: cultural identity is lost or seriously distorted.\n- 0.25--0.45: major substitutions or distortions weaken the intended identity.\n- 0.50--0.65: the broad identity remains, but important cultural meaning is weakened.\n- 0.70--0.85: cultural identity is mostly preserved, with minor inaccuracies.\n- 0.90--1.00: cultural identity and meaning are preserved without meaningful distortion.\n\nGROUP PROCEDURE\n\nYou will receive one authentic reference image and five independently generated variants,\neach paired with its current caption. Score every variant independently. Do not assume a\nfixed score distribution and do not compare the variants merely to rank them.\n\nDIRECT-SCORING REQUIREMENT\n\nMake one holistic judgment for each target. Do not generate statements, questions, cultural\nanchors, submetric scores, or a feature vector. The numeric score is the prediction.\n\nOUTPUT JSON\n\n{\n  "items": [\n    {\n      "id": "TARGET_INSTANCE_ID",\n      "score": 0.0,\n      "confidence": 0.0,\n      "visible_evidence": "one short visible observation",\n      "brief_reason": "one short sentence"\n    }\n  ]\n}\n\nReturn JSON only with exactly one item per target variant. Scores and confidence must be\nbetween 0 and 1. Do not output chain-of-thought and do not score any other CRAI metric.'.strip()

def short_hash(value: str, length: int = 12) -> str:
    return hashlib.sha256(value.strip().encode('utf-8')).hexdigest()[:length]

PROMPT_HASH = short_hash(SYSTEM_PROMPT)
print('Prompt version:', PROMPT_VERSION)
print('Prompt hash:', PROMPT_HASH)


In [ ]:
def unit_float(value, name: str) -> float:
    value = float(value)
    if not np.isfinite(value) or not 0.0 <= value <= 1.0:
        raise ValueError(f'{name} must be finite and in [0,1], got {value!r}')
    return value

def parse_json_object(text: str) -> dict:
    text = text.strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*', '', text)
        text = re.sub(r'\s*```$', '', text)
    start, end = text.find('{'), text.rfind('}')
    if start < 0 or end <= start:
        raise ValueError('No JSON object found')
    return json.loads(text[start:end + 1])

def validate_item(value: dict) -> dict:
    required = {'id', 'score', 'confidence', 'visible_evidence', 'brief_reason'}
    missing = required - set(value)
    if missing:
        raise ValueError(f'Missing fields: {sorted(missing)}')
    result = {
        'id': str(value['id']),
        'score': unit_float(value['score'], 'score'),
        'confidence': unit_float(value['confidence'], 'confidence'),
        'visible_evidence': str(value['visible_evidence']).strip(),
        'brief_reason': str(value['brief_reason']).strip(),
    }
    if not result['visible_evidence'] or not result['brief_reason']:
        raise ValueError('Evidence and reason must not be empty')
    return result

def validate_group_response(value: dict, expected_ids: list[str]) -> dict:
    items = value.get('items')
    if not isinstance(items, list) or len(items) != len(expected_ids):
        raise ValueError(f'Expected {len(expected_ids)} items')
    by_id = {}
    for raw in items:
        item = validate_item(raw)
        if item['id'] in by_id:
            raise ValueError(f'Duplicate response ID {item["id"]}')
        by_id[item['id']] = item
    if set(by_id) != set(map(str, expected_ids)):
        raise ValueError('Response IDs do not match the target group')
    return {'items': [by_id[str(value)] for value in expected_ids]}

def image_to_data_url(path: str | Path) -> str:
    path = Path(path)
    mime = {'.png': 'image/png', '.jpg': 'image/jpeg',
            '.jpeg': 'image/jpeg', '.webp': 'image/webp'}[path.suffix.lower()]
    encoded = base64.b64encode(path.read_bytes()).decode('utf-8')
    return f'data:{mime};base64,{encoded}'

def cache_tag() -> str:
    return (f'{PROMPT_VERSION}_{MODEL}_reasoning-{REASONING_EFFORT}_'
            f'detail-{IMAGE_DETAIL}_prompt-{PROMPT_HASH}_'
            f'schema-{short_hash(SCHEMA_VERSION)}')

def cache_path(split: str) -> Path:
    return CACHE_DIR / f'{METRIC.lower()}_{split}_{cache_tag()}.jsonl'

def attempt_path(split: str) -> Path:
    return CACHE_DIR / f'{METRIC.lower()}_{split}_attempts_{cache_tag()}.jsonl'

def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    records = []
    with path.open('r', encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except Exception as exc:
                raise ValueError(f'Invalid JSONL at {path}:{line_number}') from exc
    return records

def append_jsonl(path: Path, record: dict):
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')
        handle.flush()

client = None
def get_client():
    global client
    if client is None:
        key = userdata.get('openai') if IN_COLAB else os.environ.get('OPENAI_API_KEY')
        if not key:
            raise RuntimeError(
                'OpenAI key not found. In Colab, create the secret named openai.'
            )
        client = OpenAI(api_key=key)
    return client


In [ ]:
def build_group_content(group_rows: pd.DataFrame) -> list[dict]:
    group_rows = group_rows.sort_values('caption_version')
    base_id = str(group_rows['base_id'].iloc[0])
    content = [
        {'type': 'input_text',
         'text': f'TARGET GROUP {base_id}. Score all five variants independently.'},
        {'type': 'input_text', 'text': 'AUTHENTIC REFERENCE IMAGE:'},
        {'type': 'input_image',
         'image_url': image_to_data_url(group_rows['ref_image_path'].iloc[0]),
         'detail': IMAGE_DETAIL},
    ]
    for _, row in group_rows.iterrows():
        content.extend([
            {'type': 'input_text',
             'text': f"TARGET {row['id']} | CURRENT CAPTION:\n{caption_text(row)}"},
            {'type': 'input_text',
             'text': f"GENERATED IMAGE FOR {row['id']}:"},
            {'type': 'input_image',
             'image_url': image_to_data_url(row['generated_image_path']),
             'detail': IMAGE_DETAIL},
        ])
    ids = group_rows['id'].astype(str).tolist()
    content.append({
        'type': 'input_text',
        'text': f'Return JSON only. Required target IDs in order: {ids}'
    })
    return content

def call_group_judge(group_rows: pd.DataFrame, split: str) -> dict:
    group_rows = group_rows.sort_values('caption_version').copy()
    base_id = str(group_rows['base_id'].iloc[0])
    expected_ids = group_rows['id'].astype(str).tolist()
    content = build_group_content(group_rows)
    for attempt in range(1, MAX_RETRIES + 1):
        timestamp = datetime.now(timezone.utc).isoformat()
        try:
            response = get_client().responses.create(
                model=MODEL,
                reasoning={'effort': REASONING_EFFORT},
                input=[
                    {'role': 'system', 'content': [
                        {'type': 'input_text', 'text': SYSTEM_PROMPT}
                    ]},
                    {'role': 'user', 'content': content},
                ],
            )
            parsed = validate_group_response(
                parse_json_object(response.output_text), expected_ids
            )
            record = {
                'metric': TARGET_COLUMN,
                'base_id': base_id,
                'instance_ids': expected_ids,
                'split': split,
                'prompt_version': PROMPT_VERSION,
                'prompt_hash': PROMPT_HASH,
                'schema_version': SCHEMA_VERSION,
                'model': MODEL,
                'reasoning_effort': REASONING_EFFORT,
                'image_detail': IMAGE_DETAIL,
                'created_utc': timestamp,
                'response': parsed,
            }
            append_jsonl(attempt_path(split), {
                **record, 'attempt': attempt,
                'raw_response': response.output_text
            })
            return record
        except Exception as exc:
            append_jsonl(attempt_path(split), {
                'metric': TARGET_COLUMN,
                'base_id': base_id,
                'split': split,
                'attempt': attempt,
                'created_utc': timestamp,
                'error_type': type(exc).__name__,
                'error': str(exc),
            })
            if attempt == MAX_RETRIES:
                raise
            time.sleep(2 ** attempt)

def validate_cache_record(record: dict, expected_ids: list[str],
                          split: str) -> dict:
    expected = {
        'metric': TARGET_COLUMN,
        'split': split,
        'prompt_version': PROMPT_VERSION,
        'prompt_hash': PROMPT_HASH,
        'schema_version': SCHEMA_VERSION,
        'model': MODEL,
        'reasoning_effort': REASONING_EFFORT,
        'image_detail': IMAGE_DETAIL,
    }
    for key, value in expected.items():
        if record.get(key) != value:
            raise ValueError(
                f'Incompatible cache field {key!r} for {record.get("base_id")}'
            )
    copy = dict(record)
    copy['response'] = validate_group_response(
        record['response'], expected_ids
    )
    return copy

def load_or_infer_groups(frame: pd.DataFrame, split: str,
                         run_calls: bool) -> list[dict]:
    path = cache_path(split)
    cached = {}
    for record in load_jsonl(path):
        base_id = str(record.get('base_id'))
        expected_ids = frame.loc[
            frame['base_id'].astype(str).eq(base_id)
        ].sort_values('caption_version')['id'].astype(str).tolist()
        if expected_ids:
            cached[base_id] = validate_cache_record(
                record, expected_ids, split
            )

    group_ids = sorted(frame['base_id'].astype(str).unique())
    print(f'{split} valid group cache: '
          f'{sum(group in cached for group in group_ids)}/{len(group_ids)}')
    if run_calls:
        for base_id in tqdm(group_ids, desc=f'{METRIC} {split} groups'):
            if base_id not in cached:
                group_rows = frame.loc[
                    frame['base_id'].astype(str).eq(base_id)
                ].copy()
                if len(group_rows) != 5:
                    raise ValueError(f'Expected five variants for {base_id}')
                record = call_group_judge(group_rows, split)
                append_jsonl(path, record)
                cached[base_id] = record
                time.sleep(REQUEST_SLEEP_SECONDS)
    return [cached[group] for group in group_ids if group in cached]

def records_to_frame(records: list[dict]) -> pd.DataFrame:
    rows = []
    for record in records:
        response = validate_group_response(
            record['response'], record['instance_ids']
        )
        for item in response['items']:
            rows.append({
                'id': item['id'],
                'raw_score': item['score'],
                'confidence': item['confidence'],
                'visible_evidence': item['visible_evidence'],
                'brief_reason': item['brief_reason'],
            })
    result = pd.DataFrame(rows)
    if len(result):
        assert result['id'].is_unique
    return result

print('Cache family:', cache_tag())


In [ ]:
dev_records = load_or_infer_groups(dev_df, 'dev', RUN_DEV_API_CALLS)
dev_evaluation_complete = False
dev_metrics = pd.DataFrame()

if len(dev_records) == dev_df['base_id'].nunique():
    dev_predictions = records_to_frame(dev_records)
    dev_predictions[TARGET_COLUMN] = dev_predictions['raw_score']

    gold = pd.read_csv(DATA_DIR / 'dev' / 'gold_human.tsv', sep='	')
    gold['id'] = gold['id'].astype(str)
    if TARGET_COLUMN not in gold.columns:
        raise ValueError(f'{TARGET_COLUMN} missing from dev/gold_human.tsv')
    dev_evaluation = gold[['id', TARGET_COLUMN]].merge(
        dev_predictions, on='id', how='inner',
        suffixes=('_gold', '_prediction'), validate='one_to_one'
    )
    if len(dev_evaluation) != len(dev_df):
        raise ValueError('Development prediction/gold row mismatch')

    gold_column = f'{TARGET_COLUMN}_gold'
    prediction_column = f'{TARGET_COLUMN}_prediction'
    correlation = spearmanr(
        dev_evaluation[gold_column],
        dev_evaluation[prediction_column]
    ).statistic
    dev_metrics = pd.DataFrame([{
        'system': 'Raw direct GPT-5.4',
        'metric': TARGET_COLUMN,
        'spearman': float(correlation),
        'mae': float(np.mean(np.abs(
            dev_evaluation[gold_column]
            - dev_evaluation[prediction_column]
        ))),
        'n': len(dev_evaluation),
    }])
    display(dev_metrics.round(4))

    dev_predictions[[
        'id', TARGET_COLUMN, 'confidence',
        'visible_evidence', 'brief_reason'
    ]].to_csv(
        OUTPUT_DIR / f'{METRIC.lower()}_dev_direct_predictions.tsv',
        sep='	', index=False
    )
    dev_metrics.to_csv(
        OUTPUT_DIR / f'{METRIC.lower()}_dev_direct_metrics.tsv',
        sep='	', index=False
    )
    dev_evaluation_complete = True
else:
    print(f'Dev incomplete: {len(dev_records)}/'
          f'{dev_df["base_id"].nunique()} groups')


In [ ]:
if RUN_TEST_API_CALLS and not dev_evaluation_complete:
    raise RuntimeError('Test is blocked until direct dev evaluation completes.')
if RUN_TEST_API_CALLS and not test_available:
    raise RuntimeError('No test/captions.tsv was found.')

test_records = (
    load_or_infer_groups(test_df, 'test', RUN_TEST_API_CALLS)
    if test_available else []
)

if (test_available and dev_evaluation_complete
        and len(test_records) == test_df['base_id'].nunique()):
    test_predictions = records_to_frame(test_records)
    test_predictions[TARGET_COLUMN] = test_predictions['raw_score']
    test_output = test_predictions[['id', TARGET_COLUMN]].copy()
    assert len(test_output) == len(test_df)
    assert test_output['id'].is_unique
    assert set(test_output['id']) == set(test_df['id'])
    assert test_output[TARGET_COLUMN].between(0, 1).all()
    test_output.to_csv(
        OUTPUT_DIR / f'test_{METRIC.lower()}_direct_predictions.tsv',
        sep='	', index=False
    )
    print('Saved raw direct test predictions:', len(test_output))
elif test_available:
    print(f'Test incomplete: {len(test_records)}/'
          f'{test_df["base_id"].nunique()} groups')
